# Correlation Analysis — Global Speech-Graph Metrics × Perceived Ease

**Objective**: Examine correlations between global speech-graph metrics (ASPL, mean degree, LSCC size) and Cayouette perceived-ease scores (total).

**Data**: All cleaned verbatims (excluding `{}`), from `verbatim_separateurs_Laurence_{}.csv`.

**Scope**:
- Global metrics only (no windowed `sgw_*` metrics)
- LSCC ratio excluded from the analysis
- Only perceived ease as the behavioural outcome
- All verbatims included (no linear filter)

---
## Table of Contents

- [0. Common Participants between the Two Datasets](#0.-Common-Participants-between-the-Two-Datasets)
- [1. Loading Verbatims](#1.-Loading-Verbatims)
- [2. Verbatim Cleaning](#2.-Verbatim-Cleaning)
- [3. Speech-Graph Metric Computation](#3.-Speech-Graph-Metric-Computation)
- [4. Merge with Cayouette Data](#4.-Merge-with-Cayouette-Data)
- [5. Participant-Level Aggregation](#5.-Participant-Level-Aggregation)
- [6. Correlations: Global Metrics × Total Perceived Ease](#6.-Correlations:-Global-Metrics-×-Total-Perceived-Ease)
- [7. Correlation Heatmap](#7.-Correlation-Heatmap)
- [8. Scatter Plots — Significant Correlations](#8.-Scatter-Plots-—-Significant-Correlations)
- [9. Correlations by Group](#9.-Correlations-by-Group)
- [10. Scatter Plots by Group](#10.-Scatter-Plots-by-Group)
- [Results](#Results)

In [ ]:
# Standard library
from pathlib import Path

# Data & numerics
import pandas as pd
import numpy as np

# Visualisation
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import pearsonr, spearmanr

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Paths
ROOT       = Path('../').resolve()
SG2_PATH   = ROOT / 'SpeechGraph' / 'data_SpeechGraph2.csv'
OUTPUT_DIR = ROOT / 'resultats'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Imports OK')

## 0. Common Participants between the Two Datasets

In [ ]:
id_map = pd.read_csv(ROOT / 'data/id_mapping.csv')
db = pd.read_csv(ROOT / 'data/Database_Cayouette_Borealis_3.csv', sep=';', encoding='latin-1')
sg2 = pd.read_csv(SG2_PATH, sep=';')
sg2.columns = sg2.columns.str.strip()
sg2 = sg2.rename(columns={'No anonyme': 'No'})
sg2['No'] = sg2['No'].astype(str).str.strip()
verbatim_nos = sg2['No'].dropna().unique()

db_with_no = db.merge(id_map, on='id_databank', how='left')
db_in_verbatim = db_with_no[db_with_no['No'].isin(verbatim_nos)]

print('=' * 60)
print('COMMON PARTICIPANTS — Cayouette Database × Verbatims')
print('=' * 60)
print(f'  Participants in Database_Cayouette   : {len(db)}')
print(f'  Unique participants in verbatims     : {len(verbatim_nos)}')
print(f'  In common (have a verbatim)          : {len(db_in_verbatim)}')
print()
print('Breakdown by group:')
print(db_in_verbatim['Group'].value_counts().to_string())
print()
print('Breakdown by diagnosis (Group=100):')
print(db_in_verbatim[db_in_verbatim['Group'] == 100]['dx'].value_counts().to_string())

## 1. Loading Verbatims

In [ ]:
df_verbatim = sg2[sg2['n_tokens'] > 0].copy()
print(f'Non-empty verbatims: {len(df_verbatim)}')
print(f'Unique participants: {df_verbatim["No"].nunique()}')

## 2. Verbatim Cleaning

In [ ]:
df_analysis = df_verbatim.copy()
print(f'Verbatims: {len(df_analysis)}')
print(f'Unique participants: {df_analysis["No"].nunique()}')

## 3. Speech-Graph Metric Computation

In [ ]:
# Metrics loaded from precomputed CSV — rename to sg_* naming convention
df_metrics = df_analysis.rename(columns={
    'LSCC': 'sg_lscc_size',
    'AD':   'sg_avg_degree',
    'ASPL': 'sg_aspl',
}).copy()

print('Preview of global metrics:')
df_metrics[['No', 'Groupe', 'BD', 'Niveau', 'sg_aspl', 'sg_avg_degree', 'sg_lscc_size']].head(8)

## 4. Merge with Cayouette Data

In [ ]:
CAYOUETTE_COLS = [
    'id_databank', 'Group',
    'Moyenne_facilite_totale',
    'Moyenne_facilite_niveau1', 'Moyenne_facilite_niveau2', 'Moyenne_facilite_niveau3',
]

df_metrics['id_databank'] = df_metrics['No'].map(id_map.set_index('No')['id_databank'])
print(f'Verbatims with mapped id_databank: {df_metrics["id_databank"].notna().sum()}/{len(df_metrics)}')

df_merged = df_metrics.merge(db[CAYOUETTE_COLS], on='id_databank', how='left')
print(f'Rows with available ease score: {df_merged["Moyenne_facilite_totale"].notna().sum()}')

## 5. Participant-Level Aggregation

In [ ]:
SG_METRICS = ['sg_aspl', 'sg_avg_degree', 'sg_lscc_size']

CAYOUETTE_SCORES = [
    'Moyenne_facilite_totale',
    'Moyenne_facilite_niveau1', 'Moyenne_facilite_niveau2', 'Moyenne_facilite_niveau3',
]

SG_LABELS = {
    'sg_aspl': 'ASPL',
    'sg_avg_degree': 'Mean Degree',
    'sg_lscc_size': 'LSCC Size',
}

# Keep group label for colouring in figures
group_per_part = df_merged.groupby('No')['Groupe'].first()

df_per_part = (
    df_merged.groupby('No')[SG_METRICS + CAYOUETTE_SCORES]
    .mean()
    .reset_index()
)
df_per_part['Groupe'] = df_per_part['No'].map(group_per_part)
df_per_part['group_label'] = df_per_part['Groupe'].map({300.0: 'Control', 100.0: 'SSD'})

df_analysis_corr = df_per_part.dropna(subset=['Moyenne_facilite_totale']).copy()

print(f'Total participants: {len(df_per_part)}')
print(f'Participants with ease score: {len(df_analysis_corr)}')
print(f'\nDescriptive statistics:')
df_analysis_corr[SG_METRICS + ['Moyenne_facilite_totale']].describe().round(3)

## 6. Correlations: Global Metrics × Total Perceived Ease

In [ ]:
TARGET_CAYOUETTE = ['Moyenne_facilite_totale']

# Display labels for perceived-ease columns (used in plots and tables)
CAYOUETTE_LABELS = {
    'Moyenne_facilite_totale': 'Ease (total)',
    'Moyenne_facilite_niveau1': 'Ease N1',
    'Moyenne_facilite_niveau2': 'Ease N2',
    'Moyenne_facilite_niveau3': 'Ease N3',
}

corr_results = []
for sg in SG_METRICS:
    for cay in TARGET_CAYOUETTE:
        sub = df_analysis_corr[[sg, cay]].dropna()
        if len(sub) < 5:
            continue
        r_p, p_p = stats.pearsonr(sub[sg], sub[cay])
        r_s, p_s = stats.spearmanr(sub[sg], sub[cay])
        corr_results.append({
            'sg_metric': sg, 'cayouette': cay,
            'sg_label': SG_LABELS[sg], 'cay_label': CAYOUETTE_LABELS[cay],
            'pearson_r': r_p, 'pearson_p': p_p,
            'spearman_r': r_s, 'spearman_p': p_s,
            'n': len(sub)
        })

df_corr = pd.DataFrame(corr_results)

print(f"{'Metric':<14} {'Ease score':<22} {'r (Pearson)':>12} {'p':>8} {'ρ (Spearman)':>14} {'p':>8}  n")
print('-' * 85)
for _, row in df_corr.iterrows():
    sig_p = '*' if row['pearson_p'] < 0.05 else ' '
    sig_s = '*' if row['spearman_p'] < 0.05 else ' '
    print(f"{row['sg_label']:<14} {row['cay_label']:<22} {row['pearson_r']:>+11.3f}{sig_p} {row['pearson_p']:>8.4f} {row['spearman_r']:>+13.3f}{sig_s} {row['spearman_p']:>8.4f}  {row['n']}")

## 7. Correlation Heatmap

In [ ]:
mpl.rcParams['font.family'] = 'Times New Roman'

pivot_r = df_corr.pivot(index='sg_label', columns='cay_label', values='spearman_r')
pivot_p = df_corr.pivot(index='sg_label', columns='cay_label', values='spearman_p')

rename_map = {'ASPL': 'ASPL', 'Mean Degree': 'AD', 'LSCC Size': 'LSCC'}
pivot_r = pivot_r.rename(index=rename_map)
pivot_p = pivot_p.rename(index=rename_map)
# Shorten column label for display space
pivot_r.columns = [c.replace('Ease (total)', 'Ease') for c in pivot_r.columns]
pivot_p.columns = [c.replace('Ease (total)', 'Ease') for c in pivot_p.columns]

annot = pivot_r.round(2).astype(str)
for i in range(pivot_p.shape[0]):
    for j in range(pivot_p.shape[1]):
        stars = '***' if pivot_p.iloc[i, j] < .001 else '**' if pivot_p.iloc[i, j] < .01 else '*' if pivot_p.iloc[i, j] < .05 else ''
        if stars:
            annot.iloc[i, j] += f'\n{stars}'

fig, ax = plt.subplots(figsize=(4.5, 4.0))
sns.heatmap(
    pivot_r, annot=annot, fmt='', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0,
    ax=ax, linewidths=0.6, linecolor='white',
    cbar_kws={'label': "Spearman ρ", 'shrink': 0.85},
    annot_kws={'size': 20},
)
ax.set_xlabel('', labelpad=4)
ax.set_ylabel('', labelpad=4)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=20, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20, rotation=0)
ax.tick_params(length=0)
ax.figure.axes[-1].tick_params(labelsize=20)
ax.figure.axes[-1].set_ylabel("Spearman ρ", fontsize=20)

plt.tight_layout()
plt.show()

## 8. Scatter Plots — Significant Correlations

In [ ]:
COLORS  = {'Control': '#FF8C42', 'SSD': '#E63946'}
MARKERS = {'Control': 'o',       'SSD': 's'}

metrics_ordered = [
    ('sg_aspl',       'ASPL',  'A'),
    ('sg_avg_degree', 'AD',    'B'),
    ('sg_lscc_size',  'LSCC',  'C'),
]

fig, axes = plt.subplots(1, 3, figsize=(10, 4.5))

for ax, (sg, lbl, panel) in zip(axes, metrics_ordered):
    row = df_corr[df_corr['sg_metric'] == sg].iloc[0]
    sub = df_analysis_corr[[sg, 'Moyenne_facilite_totale', 'group_label']].dropna()
    sub = sub.copy()
    sub[sg] = (sub[sg] - sub[sg].mean()) / sub[sg].std()

    for grp in ['Control', 'SSD']:
        g = sub[sub['group_label'] == grp]
        ax.scatter(g[sg], g['Moyenne_facilite_totale'],
                   color=COLORS[grp], marker=MARKERS[grp],
                   s=45, alpha=0.75, edgecolors='white', linewidth=0.4,
                   label=grp, zorder=3)

    m, b = np.polyfit(sub[sg], sub['Moyenne_facilite_totale'], 1)
    x_line = np.linspace(sub[sg].min(), sub[sg].max(), 200)
    ax.plot(x_line, m * x_line + b, color='#333333', linewidth=1.8, zorder=4)

    stars    = '***' if row['pearson_p']  < .001 else '**' if row['pearson_p']  < .01 else '*' if row['pearson_p']  < .05 else ''
    stars_sp = '***' if row['spearman_p'] < .001 else '**' if row['spearman_p'] < .01 else '*' if row['spearman_p'] < .05 else ''
    ax.set_xlabel(lbl, fontsize=20, labelpad=5)
    ax.set_ylabel('Perceived ease (total)', fontsize=20, labelpad=5)
    ax.tick_params(labelsize=20)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_linewidth(0.8)

    r_line   = f"r = {row['pearson_r']:+.2f}{stars}"
    rho_line = f"ρ = {row['spearman_r']:+.2f}{stars_sp}"
    ax.text(0.5, 1.04, r_line + chr(10) + rho_line,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=20, color='#333333', clip_on=False)

fig.tight_layout(w_pad=2.5)
plt.show()

## 9. Correlations by Group

In [ ]:
rows = []
for grp_label, grp_val in [('SSD', 100.0), ('Control', 300.0)]:
    sub_grp = df_analysis_corr[df_analysis_corr['Groupe'] == grp_val]
    n = len(sub_grp)
    for sg in SG_METRICS:
        valid = sub_grp[[sg, 'Moyenne_facilite_totale']].dropna()
        r_p, p_p = pearsonr(valid[sg], valid['Moyenne_facilite_totale'])
        r_s, p_s = spearmanr(valid[sg], valid['Moyenne_facilite_totale'])
        rows.append({
            'Group': grp_label,
            'n': len(valid),
            'Metric': SG_LABELS[sg],
            'r (Pearson)': round(r_p, 3),
            'p_pearson': round(p_p, 4),
            'ρ (Spearman)': round(r_s, 3),
            'p_spearman': round(p_s, 4),
        })

df_by_group = pd.DataFrame(rows)

def fmt_p(p):
    """Format p-value with APA-style significance stars."""
    if p < 0.001: return '< .001 ***'
    if p < 0.01:  return f'{p:.3f} **'
    if p < 0.05:  return f'{p:.3f} *'
    return f'{p:.3f}'

df_display = df_by_group.copy()
df_display['p (Pearson)']  = df_by_group['p_pearson'].apply(fmt_p)
df_display['p (Spearman)'] = df_by_group['p_spearman'].apply(fmt_p)
df_display = df_display[['Group', 'n', 'Metric', 'r (Pearson)', 'p (Pearson)', 'ρ (Spearman)', 'p (Spearman)']]

print("Graph metric × total perceived ease correlations, by group\n")
print(df_display.to_string(index=False))

## 10. Scatter Plots by Group

In [ ]:
groups_ordered = [('SSD', 100.0), ('Control', 300.0)]

fig, axes = plt.subplots(2, 3, figsize=(10, 8))

for row_idx, (grp_label, grp_val) in enumerate(groups_ordered):
    sub_grp = df_analysis_corr[df_analysis_corr['Groupe'] == grp_val].copy()
    n_grp = len(sub_grp)

    for col_idx, (sg, lbl, panel) in enumerate(metrics_ordered):
        ax = axes[row_idx, col_idx]

        corr_row = df_by_group[
            (df_by_group['Group'] == grp_label) & (df_by_group['Metric'] == SG_LABELS[sg])
        ].iloc[0]

        valid = sub_grp[[sg, 'Moyenne_facilite_totale']].dropna().copy()
        valid[sg] = (valid[sg] - valid[sg].mean()) / valid[sg].std()

        ax.scatter(valid[sg], valid['Moyenne_facilite_totale'],
                   color=COLORS[grp_label], marker=MARKERS[grp_label],
                   s=45, alpha=0.75, edgecolors='white', linewidth=0.4, zorder=3)

        m, b = np.polyfit(valid[sg], valid['Moyenne_facilite_totale'], 1)
        x_line = np.linspace(valid[sg].min(), valid[sg].max(), 200)
        ax.plot(x_line, m * x_line + b, color='#333333', linewidth=1.8, zorder=4)

        def stars(p):
            return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''

        r_line   = f"r = {corr_row['r (Pearson)']:+.2f}{stars(corr_row['p_pearson'])}"
        rho_line = f"ρ = {corr_row['ρ (Spearman)']:+.2f}{stars(corr_row['p_spearman'])}"

        ax.set_xlabel(lbl, fontsize=20, labelpad=5)
        ax.set_ylabel('Perceived ease (total)', fontsize=20, labelpad=5)
        ax.tick_params(labelsize=20)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['bottom', 'left']].set_linewidth(0.8)
        ax.text(0.5, 1.04, r_line + '\n' + rho_line,
                transform=ax.transAxes, ha='center', va='bottom',
                fontsize=20, color='#333333', clip_on=False)

        if col_idx == 0:
            ax.set_ylabel(f'{grp_label} (n={n_grp})\nPerceived ease (total)', fontsize=16, labelpad=5)

fig.tight_layout(w_pad=2.5, h_pad=4.0)
plt.show()